In [ ]:
import numpy as np
import pandas as pd
from sklearn import linear_model
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# Load HousePrice.csv dataset into a pandas DataFrame
df = pd.read_csv("HousePrice.csv")

# Get basic information about the data
print(f"Original dataset shape: {df.shape}")

# Drop columns which have too many missing values (more than 50%) and Id column as well
missing_ratio = df.isna().mean()
cols_to_drop = missing_ratio[missing_ratio > 0.5].index
cols_to_drop = list(cols_to_drop) + ["Id"]
df_model = df.drop(columns=cols_to_drop)

# Group numeric and categorical features separately for preprocessing
numeric_cols = df_model.select_dtypes(include=["int64", "float64"]).columns
numeric_cols = numeric_cols.drop("SalePrice")  # exclude the target column
categorical_cols = df_model.select_dtypes(include=["object"]).columns

# Impute the missing values with median strategy for numeric features
num_imputer = SimpleImputer(strategy="median")
numeric_df = pd.DataFrame(
    num_imputer.fit_transform(df_model[numeric_cols]),
    columns=numeric_cols, index=df_model.index
)

# Impute the missing values with most_frequent strategy for categorical features
cat_imputer = SimpleImputer(strategy="most_frequent")
categorical_df = pd.DataFrame(
    cat_imputer.fit_transform(df_model[categorical_cols]),
    columns=categorical_cols, index=df_model.index
)

# Convert categorical features to dummy variables (drop_first=True to avoid multicollinearity)
dummies_categorical_df = pd.get_dummies(categorical_df, dtype=int, drop_first=True)

# Prepare features (X) and target (y)
X = pd.concat([numeric_df, dummies_categorical_df], axis=1)
y = df_model["SalePrice"]

print(f"Number of features after one-hot encoding: {X.shape[1]}")
print(f"Number of samples: {X.shape[0]}")

# Scale ALL features for Ridge/Lasso including dummy variables
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split data into train (70%) and test (30%) sets with random_state for reproducibility
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42)

# =============================================================================
# A) LINEAR REGRESSION – overfitting demonstration
# =============================================================================

# Fit Linear Regression model on train dataset
linreg_model = linear_model.LinearRegression()
linreg_model.fit(X_train, y_train)

# Generate predictions for training and test datasets
prediction_train = linreg_model.predict(X_train)
prediction_test  = linreg_model.predict(X_test)

# Evaluate and display model accuracy with Mean Absolute Error
print("\n--- A) Linear Regression ---")
print(f"Mean absolute error for training dataset: {mean_absolute_error(y_train, prediction_train):,.2f}")
print(f"Mean absolute error for test dataset    : {mean_absolute_error(y_test, prediction_test):,.2f}  ← much higher = OVERFITTING")

# Evaluate and display model accuracy with R2 Score
print(f"\nR2 score for training dataset: {r2_score(y_train, prediction_train):.4f}")
print(f"R2 score for test dataset    : {r2_score(y_test, prediction_test):.4f}  ← drops sharply = OVERFITTING")
print("Conclusion: Large gap between train/test error confirms this model is useless for prediction.")

# =============================================================================
# B) RIDGE REGRESSION (L2) – find alpha for Test MAE < $20,000
# =============================================================================

print("\n--- B) Ridge Regression (L2) ---")
# Wider log-scale grid gives a better chance of finding a good alpha
alphas = np.logspace(-2, 4, 50)

ridge_chosen = None
for a in alphas:
    ridge = linear_model.Ridge(alpha=a, max_iter=20000, tol=1e-4)
    ridge.fit(X_train, y_train)
    y_pred = ridge.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)

    if mae < 20000:
        ridge_chosen = (a, mae, ridge)
        print(f"Ridge alpha = {a:.4f}  |  Test MAE = ${mae:,.0f}  ✓")
        print(f"Train MAE   = ${mean_absolute_error(y_train, ridge.predict(X_train)):,.0f}")
        print(f"Train R²    = {r2_score(y_train, ridge.predict(X_train)):.4f}")
        print(f"Test  R²    = {r2_score(y_test, y_pred):.4f}")
        break

if ridge_chosen is None:
    print("No Ridge alpha in this grid reached MAE < $20,000. Try expanding the alpha range.")

# =============================================================================
# C) LASSO REGRESSION (L1) – find alpha for Test MAE < $20,000
# =============================================================================

print("\n--- C) Lasso Regression (L1) ---")
lasso_chosen = None
lasso_model  = None

for a in alphas:
    lasso = linear_model.Lasso(alpha=a, max_iter=200000, tol=1e-4, selection="random")
    lasso.fit(X_train, y_train)
    y_pred = lasso.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)

    if mae < 20000:
        lasso_chosen = (a, mae)
        lasso_model  = lasso
        print(f"Lasso alpha = {a:.4f}  |  Test MAE = ${mae:,.0f}  ✓")
        print(f"Train MAE   = ${mean_absolute_error(y_train, lasso.predict(X_train)):,.0f}")
        print(f"Train R²    = {r2_score(y_train, lasso.predict(X_train)):.4f}")
        print(f"Test  R²    = {r2_score(y_test, y_pred):.4f}")
        n_nonzero = np.sum(lasso_model.coef_ != 0)
        print(f"Non-zero coefficients: {n_nonzero} / {X.shape[1]}  (Lasso automatic feature selection)")
        break

if lasso_chosen is None:
    print("No Lasso alpha in this grid reached MAE < $20,000. Try expanding the alpha range.")
else:
    # Report 5 most significant continuous features
    # Continuous = original numeric columns (not one-hot encoded dummy variables)
    feature_names = X.columns
    coef_df = pd.DataFrame({"feature": feature_names, "coef": lasso_model.coef_})
    coef_df["abs_coef"] = coef_df["coef"].abs()

    top5_continuous = (
        coef_df[coef_df["feature"].isin(numeric_cols)]
        .sort_values("abs_coef", ascending=False)
        .head(5)
    )
    print("\nTop 5 most significant continuous (numeric) features by |coef| in Lasso:")
    print(f"  {'Feature':<20} {'Coefficient':>12}")
    print("  " + "-"*34)
    for _, row in top5_continuous.iterrows():
        print(f"  {row['feature']:<20} {row['coef']:>12,.2f}")


Original dataset shape: (1460, 81)


TypeError: string dtypes are not allowed, use 'object' instead